### Weighted MFCC Fingerprinting + Levenshtein Keyword Spotting / Зважений MFCC-відбиток + Левенштейн

- **Structure / Структура**: parametrization (features) + deterministic classification (metric). / параметризація (ознаки) + детерміністична класифікація (метрика).
- **Weighted features / Зважені ознаки**: пріоритизація інформативних MFCC-компонентів.
- **Distance-based decision / Рішення за відстанню**: мінімізація відстані Левенштейна між відбитками.

Formulas / Формули:
- Weighted acoustic fingerprint / Зважений акустичний відбиток:
  $$
  F = \mathrm{Serialize}\!\left( Q\!\left( \frac{1}{T} \sum_{t=1}^{T} \left(M_t \odot W\right) \right) \right)
  $$
- Deterministic classification / Детерміністична класифікація:
  $$
  W_{\mathrm{rec}} = \arg\min_{k\in V} \mathrm{Lev}\!\left(F_{\mathrm{input}}, F_k\right)
  $$

Notes / Примітки:
- Requires ffmpeg for pydub. / Потрібен ffmpeg для pydub.
- Reference filenames define keywords (e.g., start.wav → "start").


In [ ]:
# Setup: install requirements if needed (Colab/local)
# !pip install -q numpy pydub python_speech_features scikit-learn soundfile
# Note: ensure ffmpeg is installed on your system for pydub.



In [ ]:
import os
from typing import List, Dict
import numpy as np

from pydub import AudioSegment
from pydub.silence import split_on_silence

import python_speech_features as psf

# ---------------------- VOLUME NORMALIZATION ----------------------
def normalize_volume(audio: AudioSegment, target_dBFS: float = -20.0) -> AudioSegment:
    change_in_dBFS = target_dBFS - audio.dBFS
    return audio.apply_gain(change_in_dBFS)

# ---------------------- SEGMENTATION ----------------------
def split_audio_by_silence(audio_file: str, min_silence_len: int = 500, silence_thresh: int = -35, keep_silence: int = 100) -> List[AudioSegment]:
    audio = AudioSegment.from_file(audio_file)
    audio = normalize_volume(audio)
    audio_chunks = split_on_silence(
        audio_segment=audio,
        min_silence_len=min_silence_len,
        silence_thresh=silence_thresh,
        keep_silence=keep_silence
    )
    print(f"Total segments detected: {len(audio_chunks)}")
    return audio_chunks


def save_audio_segments(
    audio_file: str,
    audio_segments: List[AudioSegment]
) -> None:
    output_dir = os.path.splitext(audio_file)[0]
    os.makedirs(output_dir, exist_ok=True)

    for i, segment in enumerate(audio_segments):
        segment_filename = f"{output_dir}/segment_{i}.wav"
        segment.export(segment_filename, format="wav")
        print(f"Segment {i} saved as {segment_filename}")


def audiosegment_to_np_array(segment: AudioSegment) -> np.ndarray:
    samples = segment.get_array_of_samples()
    audio_data = np.array(samples, dtype=np.int16)
    return audio_data


def compute_mfcc_from_segment(segment: AudioSegment) -> np.ndarray:
    sample_rate = segment.frame_rate
    audio_data = audiosegment_to_np_array(segment)

    mfcc_features = psf.mfcc(
        signal=audio_data,
        samplerate=sample_rate,
        numcep=24,
        nfilt=48,
        nfft=2048
    )

    # Apply weights from MFCC[2] to MFCC[6]
    weights = np.ones(mfcc_features.shape[1])
    weights[2:8] *= 1.5
    mfcc_features *= weights

    return mfcc_features


def mfcc_to_string(mfcc_array: np.ndarray) -> str:
    averaged = np.mean(mfcc_array, axis=0)
    rounded = np.round(averaged, decimals=1)
    return ",".join(map(str, rounded))


def levenshtein_distance(s1: str, s2: str) -> int:
    if not s1:
        return len(s2)
    if not s2:
        return len(s1)

    dp = [[0] * (len(s2) + 1) for _ in range(len(s1) + 1)]

    for i in range(len(s1) + 1):
        dp[i][0] = i
    for j in range(len(s2) + 1):
        dp[0][j] = j

    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )
    return dp[len(s1)][len(s2)]


def load_keyword_dictionary(directory: str = "reference") -> Dict[str, str]:
    keyword_dict = {}

    if not os.path.isdir(directory):
        print(f"Directory not found: {directory}")
        return keyword_dict

    for filename in os.listdir(directory):
        if filename.endswith(".mp3") or filename.endswith(".wav"):
            keyword = os.path.splitext(filename)[0]
            path = os.path.join(directory, filename)
            try:
                audio = AudioSegment.from_file(path)
                mfcc = compute_mfcc_from_segment(audio)
                keyword_dict[keyword] = mfcc_to_string(mfcc)
                print(f"Loaded keyword: {keyword}")
            except Exception as e:
                print(f"Failed to process {filename}: {e}")

    return keyword_dict


def recognize_keyword(segment: AudioSegment, keyword_dict: Dict[str, str]) -> str:
    mfcc_segment = compute_mfcc_from_segment(segment)
    seg_str = mfcc_to_string(mfcc_segment)

    best_keyword = "no_keyword"
    min_distance = float("inf")
    distance_threshold = 3000

    for keyword, ref_str in keyword_dict.items():
        dist = levenshtein_distance(seg_str, ref_str)
        if dist < min_distance:
            min_distance = dist
            best_keyword = keyword

    if min_distance > distance_threshold:
        best_keyword = "no_keyword"

    return best_keyword



In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def evaluate_recognition(test_audio_file: str, expected_keywords: List[str], keyword_dict: Dict[str, str]) -> None:
    print(f"\nEvaluating {test_audio_file}...")

    segments = split_audio_by_silence(test_audio_file)
    predictions = []

    for i, segment in enumerate(segments):
        predicted = recognize_keyword(segment, keyword_dict)
        predictions.append(predicted)
        print(f"Segment {i}: predicted = {predicted}, expected = {expected_keywords[i] if i < len(expected_keywords) else 'N/A'}")

    min_len = min(len(expected_keywords), len(predictions))
    y_true = expected_keywords[:min_len]
    y_pred = predictions[:min_len]

    print("\nConfusion Matrix:")
    labels = sorted(set(y_true + y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    for label, row in zip(labels, cm):
        print(f"{label:>10}: {row}")

    print("\nMetrics:")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.2f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")
    print(f"Recall   : {recall_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")
    print(f"F1 Score : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.2f}")



In [ ]:
# Quickstart / Швидкий старт
# 1) Place reference audios into ../data/reference (e.g., start.wav)
# 2) Set a test file path below and labels list

REFERENCE_DIR = "../data/reference"
TEST_AUDIO = "../data/test_samples/test_sample.wav"  # change to your file

# Example labels (update to match your test audio):
expected_keywords = [
    "start", "stop", "left", "right", "faster", "slower", "back", "forward",
    "do", "pause", "continue", "program", "take", "put", "shutdown", "turn",
    "fly", "finish", "action", "attack",
]

# Build dictionary and run evaluation
keyword_dict = load_keyword_dictionary(REFERENCE_DIR)
if not keyword_dict:
    print("Увага: не знайдено жодного референсного аудіо для ключових слів. Перевірте шляхи.")
else:
    evaluate_recognition(TEST_AUDIO, expected_keywords, keyword_dict)

